# 🤖 เทรน G1 บน Google Colab — ได้โมเดลจริงไปใช้งาน

notebook นี้ **ต่างจาก** `explore_g1_*.ipynb` ตรงที่มัน **เทรนจริง** แล้ว
ได้ไฟล์โมเดล (`.pt`) ที่เอาไปใช้สั่งหุ่นได้

| | explore_g1_* | train_g1_colab (อันนี้) |
|---|---|---|
| ทำอะไร | ดู MDP เฉยๆ | **เทรน policy จริง** |
| ที่รัน | Mac (CPU) | **Colab (GPU ฟรี)** |
| ได้อะไร | ความเข้าใจ | **ไฟล์โมเดล .pt** |

**⚠️ ก่อนเริ่ม:** ไปที่เมนู `Runtime → Change runtime type → T4 GPU`
ให้แน่ใจว่าใช้ GPU ไม่งั้นจะช้ามาก

**หลักการ RL โดยย่อ:** เราให้ policy (neural network) ลองสั่ง action ในโลก
จำลองหลายพันตัวขนานกัน → วัด reward → อัลกอริทึม **PPO** ปรับ network ให้ได้
reward สูงขึ้น → วนซ้ำหลายรอบ (iteration) จน policy 'เก่ง'

## 1) ตรวจว่ามี GPU

ถ้าบรรทัดล่างไม่ขึ้นชื่อ GPU (เช่น Tesla T4) ให้กลับไปตั้ง Runtime ก่อน

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


## 2) ติดตั้ง mjlab

clone repo แล้วติดตั้งแบบ editable (`-e`) จะได้แก้โค้ด/เพิ่ม task ได้
(ใช้เวลาสักครู่)

In [ ]:
# clone repo
!if [ ! -d 'mjlab-custom' ]; then git clone -q https://github.com/anunpanya9/mjlab-custom.git; fi
%cd /content/mjlab-custom

import sys, os

# ติดตั้ง uv (pip ธรรมดาอ่าน [tool.uv.sources] ของ mjlab ไม่ได้ → ลง deps ไม่ครบ)
!curl -LsSf https://astral.sh/uv/install.sh | sh
UV = os.path.expanduser('~/.local/bin/uv')

# สำคัญ: ลงเข้า Python 'ตัวเดียวกับที่ kernel นี้ใช้' (sys.executable)
# ไม่งั้น uv จะลงเข้า /usr แล้ว kernel มองไม่เห็น → No module named mjlab
!{UV} pip install --python {sys.executable} -e . --index-strategy unsafe-best-match

# ยืนยัน import ได้ (ไม่ต้อง restart)
import mjlab
print('✓ ติดตั้ง mjlab เข้า', sys.executable, '— import ได้เลย')

/content/mjlab-custom
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.7/232.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.7/594.7 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741

## 3) ปิด Weights & Biases (ให้เทรนได้เลยไม่ต้อง login)

mjlab ใช้ W&B log การเทรน. ตั้ง offline เพื่อข้ามการ login (ผลเทรนยังเซฟใน
เครื่องปกติ). ถ้าอยากดู dashboard ออนไลน์ ให้ `!wandb login` แทน

In [ ]:
!wandb offline

## 4) เทรน! (นี่คือหัวใจ)

**คำสั่งเดียวจบ:** เรียก train script พร้อมพารามิเตอร์:
- `Mjlab-Velocity-Flat-Unitree-G1` — task ที่จะเทรน (G1 เดินตามคำสั่ง)
- `--env.scene.num-envs 2048` — จำลอง 2048 ตัวขนาน (ยิ่งเยอะยิ่งเรียนเร็ว
  แต่กิน VRAM; T4 ไหว ~2048–4096)
- `--agent.max-iterations 300` — เทรน 300 รอบ **(ตัวอย่างให้เห็นผลไว ~10-20
  นาที)**. ผลจริงจังใช้ 3000+ รอบ (เดินสวยขึ้นมาก แต่นานขึ้น)
- `--agent.save-interval 50` — เซฟ checkpoint ทุก 50 รอบ

**ระหว่างเทรน** ดูค่า `Mean reward` ใน log — ควร**ค่อยๆ เพิ่มขึ้น** นั่นคือ
สัญญาณว่า policy กำลังเรียนรู้ที่จะเดินตามคำสั่ง

In [ ]:
!python -m mjlab.scripts.train Mjlab-Velocity-Flat-Unitree-G1 \
    --env.scene.num-envs 2048 \
    --agent.max-iterations 3000 \
    --agent.save-interval 50

## 5) หา checkpoint ที่เทรนได้ (ไฟล์โมเดล .pt)

mjlab เซฟ checkpoint ที่ `logs/rsl_rl/<experiment_name>/<run>/`. สำหรับ G1
velocity ชื่อ experiment คือ `g1_velocity`. เราหา run ล่าสุดและ checkpoint
รอบสูงสุด

In [ ]:
import os
from pathlib import Path

log_dir = Path("/content/mjlab-custom/logs/rsl_rl/g1_velocity")
runs = sorted(log_dir.glob("*"), key=os.path.getmtime, reverse=True)
assert runs, "ไม่พบ run — เทรนสำเร็จหรือยัง?"
latest = runs[0]
ckpts = sorted(
  latest.glob("model_*.pt"), key=lambda p: int("".join(filter(str.isdigit, p.stem)))
)
checkpoint = str(ckpts[-1])
print("run ล่าสุด :", latest.name)
print("checkpoints:", [c.name for c in ckpts])
print("เลือกอันสูงสุด:", checkpoint)

## 6) ทดสอบโมเดล — ให้ policy ที่เทรนแล้วสั่งหุ่น แล้วอัดวิดีโอ

**แนวคิด:** โหลด checkpoint กลับเข้ามาเป็น policy แล้วให้มันสั่งหุ่นจริง (ไม่ใช่
random แล้ว!) เราเรนเดอร์ทีละเฟรมเองเป็นวิดีโอ — วิธีนี้ควบคุมได้เต็มที่และ
**ไม่ค้าง** (ไม่เปิด viewer ที่ Colab ไม่มีจอ)

> ตั้ง `MUJOCO_GL=egl` เพื่อเรนเดอร์แบบ headless (ไม่ต้องมีจอ) — ต้องตั้ง
> **ก่อน** import mujoco/สร้าง env ครั้งแรก

In [ ]:
os.environ["MUJOCO_GL"] = "egl"  # headless render บน Colab

from dataclasses import asdict

import imageio
import torch

import mjlab.tasks  # noqa: F401
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import RslRlVecEnvWrapper
from mjlab.rl.runner import MjlabOnPolicyRunner
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls

TASK = "Mjlab-Velocity-Flat-Unitree-G1"
device = "cuda" if torch.cuda.is_available() else "cpu"

# สร้าง env แบบ play (1 ตัว) พร้อม render_mode
env_cfg = load_env_cfg(TASK, play=True)
env_cfg.scene.num_envs = 1
eval_env = ManagerBasedRlEnv(cfg=env_cfg, device=device, render_mode="rgb_array")

# โหลด policy จาก checkpoint
agent_cfg = load_rl_cfg(TASK)
runner_cls = load_runner_cls(TASK) or MjlabOnPolicyRunner
wrapped = RslRlVecEnvWrapper(eval_env, clip_actions=agent_cfg.clip_actions)
runner = runner_cls(wrapped, asdict(agent_cfg), device=device)
runner.load(checkpoint, load_cfg={"actor": True}, strict=True, map_location=device)
policy = runner.get_inference_policy(device=device)
print("✓ โหลด policy จาก", Path(checkpoint).name)

In [ ]:
# rollout: ให้ policy สั่งหุ่น 200 step แล้วเก็บเฟรมเป็นวิดีโอ
obs = wrapped.get_observations()
frames = []
for step in range(200):
  with torch.inference_mode():
    action = policy(obs)
  obs, _, _, _ = wrapped.step(action)
  frames.append(eval_env.render())  # rgb array (H, W, 3)

out = "/content/g1_trained.mp4"
imageio.mimsave(out, frames, fps=30)
print(f"✓ อัดวิดีโอ {len(frames)} เฟรม -> {out}")

In [ ]:
from IPython.display import Video

Video("/content/g1_trained.mp4", embed=True, width=480)

## 7) ⬇️ ดาวน์โหลดโมเดลไปใช้งาน

ไฟล์ `.pt` นี้คือ **โมเดลที่ใช้งานได้จริง** — เอาไปโหลดที่เครื่องอื่น
(ที่มี mjlab) แล้วสั่งหุ่นด้วย `play --checkpoint-file <ไฟล์>` ได้เลย

In [ ]:
from google.colab import files

print("กำลังดาวน์โหลด:", checkpoint)
files.download(checkpoint)

## 8) สรุป + ทำต่อ

คุณเพิ่ง**เทรนโมเดล RL จริง**และได้ไฟล์ `.pt` ไปใช้งาน 🎉

**เอาโมเดลไปใช้ที่เครื่องตัวเอง** (ที่มี mjlab):
```bash
uv run play Mjlab-Velocity-Flat-Unitree-G1 --checkpoint-file model.pt
```

**อยากให้หุ่นเดินสวยขึ้น?** เพิ่ม `--agent.max-iterations` เป็น 3000–10000
(นานขึ้นแต่ผลดีขึ้นมาก) แล้วเทรนใหม่

**อยากเทรนงานหยิบของแทน?** เปลี่ยน task เป็น `Mjlab-Lift-Cube-G1`
(experiment_name = `g1_lift_cube`) แล้วแก้ path ใน cell ที่ 5 ตามนั้น

**เข้าใจว่าข้างในทำงานยังไง?** กลับไปดู `explore_g1_velocity.ipynb` และ
`explore_g1_manipulation.ipynb` ที่แกะ MDP ทีละส่วน